PUBLIC RELEASE: Original Endek photo and crop gallery outputs have been redacted for image rights. Notebook source cells regenerate these visualizations from the private local dataset. No V2 GAN weights or fabricated model metrics are included.

# ENDEKGAN-T2I V2 — FULL TRAINING WITH THE NEW PREPROCESSED DATASET

**Notebook eksperimen training ulang yang sebenarnya (bukan notebook pemeriksaan semata).**

**Dataset utama yang ditetapkan untuk eksperimen:** `full_plus_crop_ablation_B`, **1.280 file training**, **37 validation**, **37 testing**, total **1.354 file hasil preprocessing**. Kelas flora, fauna, dekoratif, geometris masing-masing memiliki 320 file training. Perlakuan ini menggabungkan `full-image resize`, *photometric augmentation* dan 336 *crop candidates*.

**Dataset pembanding:** `full_image_augmented_A`, 944 file training + validation/test yang sama. Kode asli untuk generator, discriminator, conditioning MiniLM, feature matching, R1, EMA dan *seam-aware loss* tetap dipertahankan. **Seluruh training V2 dimulai dari bobot model baru**, bukan checkpoint penelitian lama.

**Peringatan ilmiah:** Crop pada dataset B masih dipilih menggunakan heuristik tepi/tekstur dan mempertahankan prompt gambar global. Oleh karena itu, B adalah dataset utama **eksperimen ablation cropping**, belum terbukti lebih unggul secara semantik dan belum merupakan hasil penggabungan sumber Mendeley/Roboflow. Data dan hasil training lengkap disimpan di drive D; validation/test **tidak** dicrop atau diaugmentasi.

**Petunjuk:** `Run All` aman secara default. Untuk menjalankan training berat, periksa CUDA dan VRAM lalu ubah `TRAIN_NOW=True` di sel training. Jika GPU sedang dipakai proyek lain, jangan jalankan training.

## 1. Setup dan identitas dataset training (drive D saja)

Tidak ada perubahan pada experiment lama. Notebook ini membaca hasil preprocessing baru dari folder `ENDEK_PREPROCESSING_V2` dan menggunakan engine `endek_v2_preprocessing_training_engine.py`.

In [ ]:
from pathlib import Path
import os,sys,json,subprocess,importlib.util,datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

ROOT=Path(r"D:\DISERTASI_S3\WIWIK\EXPERIMEN_2\ENDEK_T2I_EXPERIMENT_V2")
STAGED=ROOT/"TRAINING_SPLITS_PREPROCESSED_V2"
ENGINE=ROOT/"endek_v2_preprocessing_training_engine.py"
TRAINER=ROOT/"run_training_from_preprocessing_V2.py"
EVALUATOR=ROOT/"evaluate_new_EndekGAN_V2_checkpoint.py"
PROMPT_GENERATOR=ROOT/"generate_Endek_from_prompt_V2.py"
CLASSES=("flora","fauna","dekoratif","geometris")
assert all(p.is_file() for p in (ENGINE,TRAINER,EVALUATOR,PROMPT_GENERATOR))
assert sys.executable.lower().startswith("d:\\vscode-multiagent-ai")
assert sys.base_prefix.lower().startswith("d:\\vscode-multiagent-ai")
print("Kernel:",sys.executable)
print("Base Python:",sys.base_prefix)
print("Trainer:",TRAINER)
print("Dataset manifests:",STAGED)

Kernel: D:\VSCode-MultiAgent-AI\envs\endekgan-t2i-v2-py311\Scripts\python.exe
Base Python: D:\VSCode-MultiAgent-AI\runtimes\python-3.11.4
Trainer: D:\DISERTASI_S3\WIWIK\EXPERIMEN_2\ENDEK_T2I_EXPERIMENT_V2\run_training_from_preprocessing_V2.py
Dataset manifests: D:\DISERTASI_S3\WIWIK\EXPERIMEN_2\ENDEK_T2I_EXPERIMENT_V2\TRAINING_SPLITS_PREPROCESSED_V2


## 2. Audit manifest final untuk training B dan pembanding A

**Manifest training** dibentuk dari preprocessing yang sudah dijalankan, tidak memakai citra lama secara langsung dan tidak menganggap varian augmentasi sebagai motif independen baru. Periksa keberadaan file, label, hash asal, prompt dan pemisahan validation/test.

In [ ]:
def load_arm(arm,split):
    name={"B":"full_plus_crop_ablation_B","A":"full_image_augmented_A"}[arm]
    return pd.read_csv(STAGED/name/(split+"_split.csv"),encoding="utf-8-sig",keep_default_na=False)
datasets={arm:{sp:load_arm(arm,sp) for sp in ("train","val","test")} for arm in ("A","B")}
summary=[]
for arm,parts in datasets.items():
    for split,frame in parts.items():
        assert frame["image_path"].map(lambda s:Path(s).is_file()).all()
        assert frame["prompt"].astype(str).str.strip().ne("").all()
        assert frame["source_sha256"].astype(str).str.fullmatch(r"[a-f0-9]{64}").all()
        assert frame["md5"].astype(str).str.fullmatch(r"[a-f0-9]{32}").all()
        assert frame["label"].isin(CLASSES).all()
        assert frame["split"].eq(split).all()
    groups={sp:set(parts[sp].source_sha256) for sp in parts}
    assert not groups["train"] & groups["val"]
    assert not groups["train"] & groups["test"]
    assert not groups["val"] & groups["test"]
    summary.append({"Arm":arm,"Training":len(parts["train"]),"Validation":len(parts["val"]),
                    "Testing":len(parts["test"]),"Total":sum(map(len,parts.values()))})
assert summary==[
 {"Arm":"A","Training":944,"Validation":37,"Testing":37,"Total":1018},
 {"Arm":"B","Training":1280,"Validation":37,"Testing":37,"Total":1354}]
assert set(datasets["A"]["val"].source_sha256)==set(datasets["B"]["val"].source_sha256)
assert set(datasets["A"]["test"].source_sha256)==set(datasets["B"]["test"].source_sha256)
display(pd.DataFrame(summary))
print("Kelas B:",datasets["B"]["train"].label.value_counts().to_dict())

,Arm,Training,Validation,Testing,Total
0,A,944,37,37,1018
1,B,1280,37,37,1354


Kelas B: {'flora': 320, 'fauna': 320, 'dekoratif': 320, 'geometris': 320}


## 3. Contoh data latih dan kondisi cropping

Bandingkan satu contoh resized full image dengan crop dari sumber yang sama. Cropping **tidak selalu mempertahankan komposisi motif utuh**; ini bagian hipotesis eksperimen yang akan diuji.

In [ ]:
btrain=datasets["B"]["train"]
cropped=btrain[btrain["is_crop"].astype(str).str.lower().isin(["true","1"])]
assert len(cropped)==336
row=cropped.iloc[0]
baseline=btrain[(btrain.source_sha256==row.source_sha256)&(~btrain.is_crop.astype(str).str.lower().isin(["true","1"]))]
assert len(baseline)>0
first=baseline.iloc[0]
fig,axes=plt.subplots(1,2,figsize=(7,3.4))
for ax,item,title in zip(axes,[first,row],["Full-image resized","Cropped for training B"]):
    with Image.open(item.image_path) as im:ax.imshow(im.convert("RGB"))
    ax.axis("off");ax.set_title(title,fontweight="bold")
plt.tight_layout();plt.show()
print("Motif class:",row.label)
print("Source-aware split:",row.split)
print("Text prompt (inherited, NOT crop-verified):",row.prompt[:260]+"...")

Motif class: flora
Source-aware split: train
Text prompt (inherited, NOT crop-verified): Motif Endek Flora berbasis daun dan sulur tanaman tropis Bali. Desain kain Endek Bali bertema flora yang terinspirasi dari daun dan sulur tanaman tropis Bali. Motif dibuat menggunakan teknik stilasi, yaitu penggayaan bentuk asli menjadi ornamen dekoratif denga...


## 4. Protokol training yang sebanding antara A dan B

**Unit pembandingan:** jumlah `optimizer updates`, bukan jumlah epoch saja. Pada kedua dataset gunakan **512 sampel terambil per epoch** dengan sampler kelas seimbang, batch size **4**, sehingga terdapat **128 update/epoch**, identik untuk A dan B. Model yang sama dibangun dari bobot awal acak untuk setiap seed.

- **Proposed:** `seam_proposed` = Transformer MiniLM + class embedding + seam loss 0.5.
- **Baseline:** `transformer` = MiniLM + class embedding tanpa seam loss.
- **Ablation lain:** `cat`, `hash_text`, `transformer_noclass`, `seam_noclass`.
- Seed model: **42, 123, 2025**.
- Maksimum 400 epoch; checkpoint dipilih berdasarkan **validation KID**, mulai epoch 200. Testing tidak boleh dipakai untuk pemilihan checkpoint.
- *Online flip/jitter* dari notebook lama **dinonaktifkan**, karena dataset sudah memiliki perlakuan preprocessing offline.
- **Tidak ada penggunaan checkpoint lama.**

In [ ]:
MODELS=["cat","hash_text","transformer","transformer_noclass","seam_proposed","seam_noclass"]
SEEDS=[42,123,2025]
DATASET_MAIN="B"                  # FIX: full image + augmentation + crop; 1280 train
MODEL_TO_TRAIN="seam_proposed"     # Proposed architecture
SEED_TO_TRAIN=42
BATCH_SIZE=4
SAMPLES_PER_EPOCH=512
EPOCHS_FULL=400
assert SAMPLES_PER_EPOCH%BATCH_SIZE==0
internal_matrix=[
    dict(arm="B",model=model,seed=seed,status="NOT_STARTED")
    for model in MODELS for seed in SEEDS
]
control_matrix=[
    dict(arm="A",model=model,seed=seed,status="NOT_STARTED")
    for model in ("transformer","seam_proposed") for seed in SEEDS
]
matrix=pd.DataFrame(internal_matrix+control_matrix)
print("Internal planned runs B:",len(internal_matrix)," | Controls A:",len(control_matrix))
print("External adapted DF-GAN–MiniLM: 3 additional independent runs, separate original architecture/runner")
display(matrix.head(10))
matrix.to_csv(ROOT/"TRAINING_MATRIX_PREPROCESSED_V2.csv",index=False,encoding="utf-8-sig")

Internal planned runs B: 18  | Controls A: 6
External adapted DF-GAN–MiniLM: 3 additional independent runs, separate original architecture/runner


,arm,model,seed,status
0,B,cat,42,NOT_STARTED
1,B,cat,123,NOT_STARTED
2,B,cat,2025,NOT_STARTED
3,B,hash_text,42,NOT_STARTED
4,B,hash_text,123,NOT_STARTED
5,B,hash_text,2025,NOT_STARTED
6,B,transformer,42,NOT_STARTED
7,B,transformer,123,NOT_STARTED
8,B,transformer,2025,NOT_STARTED
9,B,transformer_noclass,42,NOT_STARTED


## 5. Pemeriksaan PyTorch CUDA, RTX dan dependensi pada D

**Jika ada paket yang belum terpasang**, jalankan satu kali `INSTALL_CUDA_PYTORCH_ON_D_FOR_V2.cmd` dari folder proyek, tanpa mengubah environment lainnya. Instalasi paket memerlukan koneksi jaringan serta ruang disk pada D.

CUDA sendiri tidak boleh dianggap siap hanya karena perangkat RTX terdeteksi. Training 256×256 juga memerlukan VRAM yang cukup; peluncur menolak training CUDA jika VRAM bebas di bawah **6.400 MiB**.

In [ ]:
required_modules=("torch","torchvision","sklearn","transformers","torchmetrics","torch_fidelity")
modules={m:bool(importlib.util.find_spec(m)) for m in required_modules}
try:
    gpu=subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,memory.used,memory.free",
                        "--format=csv,noheader,nounits"],
                        capture_output=True,text=True,timeout=10)
    print("RTX memory:",gpu.stdout.strip() or gpu.stderr.strip())
except Exception as e:
    print("Cannot inspect GPU:",repr(e))
print("Modules:",json.dumps(modules,indent=2))
if modules["torch"]:
    import torch
    print("torch version:",torch.__version__)
    print("CUDA available:",torch.cuda.is_available())
else:
    print("PyTorch/CUDA belum dipasang pada environment D ini.")
print("CUDA install script:",ROOT/"INSTALL_CUDA_PYTORCH_ON_D_FOR_V2.cmd")

RTX memory: NVIDIA GeForce RTX 3070 Ti Laptop GPU, 8192, 5760, 2258
Modules: {
  "torch": false,
  "torchvision": false,
  "sklearn": false,
  "transformers": false,
  "torchmetrics": false,
  "torch_fidelity": false
}
PyTorch/CUDA belum dipasang pada environment D ini.
CUDA install script: D:\DISERTASI_S3\WIWIK\EXPERIMEN_2\ENDEK_T2I_EXPERIMENT_V2\INSTALL_CUDA_PYTORCH_ON_D_FOR_V2.cmd


## 6. Dry run — pratinjau training B (tidak memulai GPU)

Memastikan jumlah data, kelas, sumber, pilihan generator/regularizer dan memori GPU. Ini berbeda dengan **training asli** pada sel selanjutnya.

In [ ]:
def build_training_command(arm,model,seed,execute=False,resume=False):
    cmd=[sys.executable,str(TRAINER),"--arm",arm,"--model",model,"--seed",str(seed),
         "--device","cuda","--batch-size",str(BATCH_SIZE),
         "--samples-per-epoch",str(SAMPLES_PER_EPOCH),"--epochs",str(EPOCHS_FULL)]
    if execute:cmd.append("--execute")
    if resume:cmd.append("--resume")
    return cmd
dryrun=subprocess.run(build_training_command(DATASET_MAIN,MODEL_TO_TRAIN,SEED_TO_TRAIN),
                      text=True,capture_output=True,timeout=45)
print(dryrun.stdout if dryrun.returncode==0 else dryrun.stderr)
assert dryrun.returncode==0
print("DRY-RUN DONE; no model trained yet.")

{
  "status": "PREFLIGHT_ONLY",
  "dataset_arm": "B",
  "split_dir": "D:\\DISERTASI_S3\\WIWIK\\EXPERIMEN_2\\ENDEK_T2I_EXPERIMENT_V2\\TRAINING_SPLITS_PREPROCESSED_V2\\full_plus_crop_ablation_B",
  "source_dataset": "full_plus_crop_ablation_B",
  "run_id": "seam_proposed_seed42",
  "model": "seam_proposed",
  "encoder": "transformer_text",
  "seam_lambda": 0.5,
  "disable_class": false,
  "seed": 42,
  "image_size": 256,
  "batch_size": 4,
  "optimizer_updates_per_epoch": 128,
  "train_samples_per_epoch": 512,
  "epochs": 400,
  "total_planned_optimizer_updates": 51200,
  "online_augmentation": false,
  "training_method": "New weights from random initialization; weighted class-balanced fixed number of samples per epoch",
  "validation_model_selection": "KID on held-out validation, no test for checkpoint selection",
  "testing": "Only after new V2 checkpoint exists",
  "checkpoint_source": "NEW_MODEL_RANDOM_INIT",
  "output_dir": "D:\\DISERTASI_S3\\WIWIK\\EXPERIMEN_2\\ENDEK_T2I_EXPERIMENT

## 7. TRAINING LANGSUNG — pilih satu konfigurasi, lalu eksekusi

**Kode di bawah merupakan TRAINING GAN sebenarnya**, bukan simulasi. Untuk memulai, ubah `TRAIN_NOW=False` menjadi `True`, lalu jalankan sel ini setelah CUDA aktif dan cukup kosong. Prosesnya dapat panjang karena setiap model dilatih dari awal.

Hasil tersimpan di `runs_preprocessed/full_plus_crop_ablation_B/seam_proposed_seed42/` (atau sesuai pilihan) berisi `training_log.csv`, `checkpoints/`, `run_state.json`, dan `experiment_plan.json`.

Jika sebelumnya terinterupsi, ubah `RESUME_EXISTING_RUN=True` untuk melanjutkan **checkpoint V2 milik run yang sama**, bukan model dari eksperimen lama.

In [ ]:
TRAIN_NOW=False              # SET TRUE to start the actual full GAN training
RESUME_EXISTING_RUN=False     # True only for existing local V2 last checkpoint

if TRAIN_NOW:
    if not all(modules.values()):
        raise RuntimeError("Install required PyTorch CUDA packages in D before full training.")
    cmd=build_training_command(DATASET_MAIN,MODEL_TO_TRAIN,SEED_TO_TRAIN,
                               execute=True,resume=RESUME_EXISTING_RUN)
    print("STARTING REAL GAN TRAINING:",cmd)
    with subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          text=True,bufsize=1,encoding="utf-8",errors="replace") as proc:
        for line in proc.stdout:
            print(line,end="",flush=True)
        code=proc.wait()
    if code!=0:
        raise RuntimeError(f"Training exited with code {code}; inspect GPU/dependencies/run log")
    print("Training completed. New checkpoint must now be independently evaluated.")
else:
    print("TRAIN_NOW=False — actual training cell prepared; GPU workloads remain untouched.")

TRAIN_NOW=False — actual training cell prepared; GPU workloads remain untouched.


## 8. Batch training untuk seluruh konfigurasi internal (opsional)

**18 run utama** pada B dan **6 run kontrol** pada A. Jalankan secara berurutan dan periksa output setiap run. Batch tidak akan dimulai otomatis. Implementasi **adapted DF-GAN–MiniLM eksternal** tetap memerlukan runner khusus karena arsitekturnya berbeda. Jangan menyebutnya selesai sebelum benar-benar dilatih.

In [ ]:
RUN_ALL_INTERNAL=False          # Change to True only if ready for sequential GPU runs
if RUN_ALL_INTERNAL:
    if not all(modules.values()):raise RuntimeError("Missing CUDA training dependencies")
    for row in matrix.itertuples(index=False):
        cmd=build_training_command(row.arm,row.model,int(row.seed),execute=True)
        print("RUN:",row.arm,row.model,row.seed,flush=True)
        completed=subprocess.run(cmd,check=False)
        if completed.returncode!=0:
            raise RuntimeError(f"STOP: training failed at {row.arm}/{row.model}/seed{row.seed}")
    print("All 24 internal/control training jobs executed. Verify their run_state.json files.")
else:
    print("Batch training disabled. Main dataset B and A controls are staged, not trained.")

Batch training disabled. Main dataset B and A controls are staged, not trained.


## 9. Pemeriksaan hasil checkpoint, grafik loss dan validasi (jika sudah training)

**Hanya menampilkan hasil nyata.** Tidak membuat angka FID/KID atau mengklaim *training complete* ketika checkpoint belum ada.

In [ ]:
ARM_DIR={"A":"full_image_augmented_A","B":"full_plus_crop_ablation_B"}
SELECTED_RUN=ROOT/"runs_preprocessed"/ARM_DIR[DATASET_MAIN]/f"{MODEL_TO_TRAIN}_seed{SEED_TO_TRAIN}"
STATE=SELECTED_RUN/"run_state.json"
LOG=SELECTED_RUN/"training_log.csv"
if STATE.is_file():
    state=json.loads(STATE.read_text(encoding="utf-8"))
    print("Run state:",json.dumps(state,indent=2,ensure_ascii=False))
else:
    print("Belum ada model GAN V2 yang selesai dilatih pada konfigurasi terpilih.")
if LOG.is_file():
    log=pd.read_csv(LOG)
    display(log.tail(5))
    ax=log.plot(x="epoch",y=["g_loss","d_loss"],figsize=(9,4),
                title="Actual training losses — EndekGAN-T2I V2")
    ax.set_ylabel("Loss (not image quality)");plt.tight_layout();plt.show()
else:
    print("Training loss graph not available because full training has not run.")

Belum ada model GAN V2 yang selesai dilatih pada konfigurasi terpilih.
Training loss graph not available because full training has not run.


## 10. Evaluasi hasil generasi model baru: FID, KID, LPIPS dan seam gap

Eksekusi evaluasi dilakukan **hanya sesudah checkpoint V2 sendiri selesai**; data test tidak digunakan untuk *checkpoint selection*. Karena test real relatif kecil, FID/KID (terutama per kelas) bersifat eksploratif dan tidak cukup sendirian untuk mengklaim peningkatan kualitas.

`EVALUATE_NOW=True` akan membuat gambar model dari prompt test dan menghitung metrik. Jangan dijalankan sebelum training berhasil.

In [ ]:
EVALUATE_NOW=False
if EVALUATE_NOW:
    cmd=[sys.executable,str(EVALUATOR),"--arm",DATASET_MAIN,
         "--model",MODEL_TO_TRAIN,"--seed",str(SEED_TO_TRAIN),
         "--device","cuda","--samples-per-class","64","--execute"]
    result=subprocess.run(cmd,check=False)
    if result.returncode!=0:raise RuntimeError("Evaluation failed; check checkpoint, torchmetrics and data.")
else:
    cmd=[sys.executable,str(EVALUATOR),"--arm",DATASET_MAIN,
         "--model",MODEL_TO_TRAIN,"--seed",str(SEED_TO_TRAIN)]
    result=subprocess.run(cmd,capture_output=True,text=True,timeout=30)
    print(result.stdout if result.returncode==0 else result.stderr)

{
  "run_dir": "D:\\DISERTASI_S3\\WIWIK\\EXPERIMEN_2\\ENDEK_T2I_EXPERIMENT_V2\\runs_preprocessed\\full_plus_crop_ablation_B\\seam_proposed_seed42",
  "expected_new_checkpoint": "D:\\DISERTASI_S3\\WIWIK\\EXPERIMEN_2\\ENDEK_T2I_EXPERIMENT_V2\\runs_preprocessed\\full_plus_crop_ablation_B\\seam_proposed_seed42\\checkpoints",
  "test_split_dir": "D:\\DISERTASI_S3\\WIWIK\\EXPERIMEN_2\\ENDEK_T2I_EXPERIMENT_V2\\TRAINING_SPLITS_PREPROCESSED_V2\\full_plus_crop_ablation_B",
  "evaluation_prompts": "HELD_OUT_TEST_ONLY",
  "samples_per_class": 64,
  "status": "PREFLIGHT_ONLY",
  "training_finished": false
}
NO TEST EVALUATION RUN: add --execute only when a completed V2 run exists



## 11. Text-to-Image yang dapat digunakan setelah training

**Penggunaan akhir:** pengguna mengetik prompt bebas → MiniLM mengode teks → Generator terlatih menghasilkan motif baru. Pilihan kelas tetap diperlukan sebagai conditioning agar konsisten dengan desain model sebelumnya.

Contoh ini **hanya akan menghasilkan gambar** jika `GENERATE_FROM_PROMPT_NOW=True` dan checkpoint V2 terpilih telah selesai.

In [ ]:
PROMPT_ENDek="Motif Endek Bali dengan bunga teratai, warna biru dan emas, teknik stilasi, pola ulang menyeluruh."
PROMPT_CLASS="flora"
LATENT_SEED=2025
GENERATE_FROM_PROMPT_NOW=False
if GENERATE_FROM_PROMPT_NOW:
    command=[sys.executable,str(PROMPT_GENERATOR),"--arm",DATASET_MAIN,
             "--model",MODEL_TO_TRAIN,"--seed",str(SEED_TO_TRAIN),
             "--prompt",PROMPT_ENDek,"--class",PROMPT_CLASS,
             "--latent-seed",str(LATENT_SEED),"--device","cuda","--execute"]
    completed=subprocess.run(command,check=False)
    if completed.returncode!=0:raise RuntimeError("Prompt generation failed; check new V2 checkpoint")
else:
    print("Prompt input configured; first complete GAN training and set GENERATE_FROM_PROMPT_NOW=True.")

Prompt input configured; first complete GAN training and set GENERATE_FROM_PROMPT_NOW=True.


## 12. Simpan protokol eksperimen dan tindak lanjut penelitian

Tidak ada hasil lama yang dianggap berasal dari dataset preprocessing baru. Eksperimen yang sah perlu membandingkan A/B dengan *fixed source split*, *matched update budget*, metrik kuantitatif dan pada akhir proses evaluasi pakar terhadap **motif yang dihasilkan AI**.

In [ ]:
plan={
    "study":"EndekGAN-T2I with new full-image + crop preprocessing V2",
    "selected_main_arm":"B",
    "selected_dataset_path":str(STAGED/"full_plus_crop_ablation_B"),
    "selected_dataset_file_count":1354,
    "train":1280,"validation":37,"test":37,
    "baseline_control_arm":"A",
    "baseline_training_count":944,
    "conditioned_classes":list(CLASSES),
    "all_internal_models":MODELS,
    "training_seeds":SEEDS,
    "batch_size":BATCH_SIZE,
    "train_samples_per_epoch":SAMPLES_PER_EPOCH,
    "optimizer_updates_per_epoch":SAMPLES_PER_EPOCH//BATCH_SIZE,
    "full_max_epochs":EPOCHS_FULL,
    "online_augmentation":False,
    "random_initialization":True,
    "external_datasets":"NOT YET INTEGRATED",
    "crop_prompt_semantics":"NOT INDEPENDENTLY VERIFIED",
    "expert_review":"AFTER GENERATED GAN IMAGES",
    "status":"TRAINING_PREPARED_NOT_STARTED" if not STATE.is_file() else "CHECK_RUN_STATE",
}
(ROOT/"FINAL_TRAINING_PROTOCOL_PREPROCESSED_V2.json").write_text(
    json.dumps(plan,indent=2,ensure_ascii=False),encoding="utf-8")
display(pd.DataFrame([{"Main dataset":plan["selected_main_arm"],
                        "Train files":plan["train"],"Validation":plan["validation"],
                        "Test":plan["test"],"Update / epoch":plan["optimizer_updates_per_epoch"],
                        "Status":plan["status"]}]))
print("Training protocol written:",ROOT/"FINAL_TRAINING_PROTOCOL_PREPROCESSED_V2.json")

,Main dataset,Train files,Validation,Test,Update / epoch,Status
0,B,1280,37,37,128,TRAINING_PREPARED_NOT_STARTED


Training protocol written: D:\DISERTASI_S3\WIWIK\EXPERIMEN_2\ENDEK_T2I_EXPERIMENT_V2\FINAL_TRAINING_PROTOCOL_PREPROCESSED_V2.json


## Kesimpulan

Dataset utama training V2 ditetapkan sebagai **B dengan 1.280 file training**; kontrol A dengan **944 file training**. Kedua perlakuan menggunakan validation/test sama dan arsitektur EndekGAN-T2I yang sama. Notebook ini **mempunyai kode training, checkpoint, evaluasi dan text-to-image inference yang nyata**, tetapi training otomatis sengaja dinonaktifkan demi keamanan RTX yang sedang digunakan.

**Untuk mulai training:** pasang dependensi CUDA pada environment D, pastikan VRAM GPU bebas cukup, ubah `TRAIN_NOW=True`, kemudian jalankan sel 7. Jangan memublikasikan hasil kuantitatif sebelum checkpoint baru dan evaluasinya benar-benar tersedia.